# ⚡ Colab Pro A100 — Fast F2 Retrieval Submission

This notebook is optimized for the official retrieval metric: macro F2, where recall is weighted 4× precision.

Key speed choices:
- Retrieval-only: `USE_GENERATOR = False` so Qwen answer generation is skipped.
- A100-friendly dense + rerank path.
- Moderate candidate breadth (`RERANK_POOL=96`) for much faster runtime than the wide Kaggle notebook.
- Optional sharding so multiple Colab/Kaggle sessions can split the test set.

Outputs:
- `/content/results.json`
- `/content/submission.zip`


## 1. Runtime + config

Use **Runtime → Change runtime type → A100 GPU** if Colab Pro offers it.

For dataset access, either:
1. Upload `kaggle.json` when prompted, or
2. Put `kaggle.json` in `/content/drive/MyDrive/kaggle.json` and set `MOUNT_DRIVE=True`.

In [ ]:
# ===== Main config ======================================================
GITHUB_REPO = 'https://github.com/vkb0205/Road2AI_ApplePie.git'
REPO_BRANCH = 'combined_recall_pipeline'
REPO_DIR = '/content/Road2AI_ApplePie'

KAGGLE_DATASET = 'vkb0205/stage6-data'
DATA_DIR = '/content/stage6-data'

# If INPUT_QUERIES is blank, the notebook will auto-detect common test files in DATA_DIR.
INPUT_QUERIES = ''

RESULTS_PATH = '/content/results.json'
SUBMISSION_ZIP_PATH = '/content/submission.zip'

# Optional sharding: run multiple notebooks with SHARD_COUNT=N and SHARD_INDEX=0..N-1.
# For a single A100 run, keep SHARD_COUNT=1.
SHARD_COUNT = 1
SHARD_INDEX = 0
SHARDED_INPUT_PATH = '/content/questions_shard.json'

# Dataset auth. Set MOUNT_DRIVE=True if kaggle.json is in Drive.
MOUNT_DRIVE = False

# ===== Fast F2 retrieval settings ======================================
USE_DENSE = True
USE_RERANK = True
USE_DECOMPOSITION = True
USE_LLM_DECOMPOSITION = False   # fastest: rule planner only; set True only if time allows
USE_GENERATOR = False           # critical speedup; retrieval F2 uses relevant_articles/docs

USE_MULTI_VARIANT = True
USE_COMPLEXITY_K = True
COVERAGE_QUOTA = True
MAX_VARIANTS = 4
MAX_SUB_QUERIES = 4

FTS_MODE = 'bm25_ranked'
GPU_ID = 0

# Retrieval breadth: faster than the wide Kaggle config, still recall-oriented.
TOP_BM25 = 300
TOP_DENSE = 140
RRF_K = 60
TOP_DOCS = 40
PER_DOC_ARTICLES = 10
RERANK_POOL = 96
CHUNKS_PER_ARTICLE = 4

# Selection: keep enough articles because F2 weights recall 4x precision.
DROP_PROVINCIAL = True
MAX_K = 10
MIN_K = 1
REL_MARGIN = 0.32
ABS_MARGIN = 0.22

GEN_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
GEN_LOAD_IN_4BIT = True
GEN_CONTEXT_TOPK = 6
# ======================================================================


## 2. Install dependencies and check GPU

In [ ]:
import os, sys, json, shutil, subprocess, zipfile, time
from pathlib import Path

gpu = subprocess.run('nvidia-smi', shell=True, text=True, capture_output=True)
print(gpu.stdout if gpu.returncode == 0 else gpu.stderr)
if gpu.returncode != 0:
    raise SystemExit('No GPU detected. In Colab: Runtime → Change runtime type → GPU.')

!pip -q install -U kaggle hf_transfer pandas pyarrow networkx pyyaml psutil tqdm
!pip -q install -U 'FlagEmbedding>=1.2.10,<1.3' 'transformers>=4.41,<4.46' accelerate bitsandbytes
# FAISS GPU wheels vary by CUDA/Python. Try GPU first, then CPU fallback.
!pip -q install faiss-gpu-cu12 || pip -q install faiss-cpu

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('deps ready')


## 3. Download Stage-6 dataset from Kaggle

In [ ]:
from pathlib import Path

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    src = Path('/content/drive/MyDrive/kaggle.json')
    if not src.exists():
        raise FileNotFoundError('Expected /content/drive/MyDrive/kaggle.json')
    Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, '/root/.kaggle/kaggle.json')
else:
    if not Path('/root/.kaggle/kaggle.json').exists():
        from google.colab import files
        print('Upload kaggle.json now...')
        uploaded = files.upload()
        if 'kaggle.json' not in uploaded:
            raise FileNotFoundError('kaggle.json was not uploaded')
        Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
        shutil.move('/content/kaggle.json', '/root/.kaggle/kaggle.json')

!chmod 600 /root/.kaggle/kaggle.json
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
!kaggle datasets download -d {KAGGLE_DATASET} -p /content --force
zip_name = '/content/' + KAGGLE_DATASET.split('/')[-1] + '.zip'
!unzip -q -o {zip_name} -d {DATA_DIR}
print('DATA_DIR files:')
for p in sorted(Path(DATA_DIR).glob('*'))[:50]:
    print(' -', p)


## 4. Clone repo and import source

In [ ]:
if not Path(REPO_DIR).exists():
    !git clone --depth 1 -b {REPO_BRANCH} {GITHUB_REPO} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch --depth 1 origin {REPO_BRANCH} && git reset --hard origin/{REPO_BRANCH}

SRC = str(Path(REPO_DIR) / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('src on path:', SRC)


## 5. Auto-detect questions and optionally shard

In [ ]:
def detect_questions_file(data_dir, explicit=''):
    if explicit and Path(explicit).exists():
        return Path(explicit)
    candidates = [
        'test.json',
        'questions.json',
        'R2AIStage1DATA.json',
        'r2ai-testset/R2AIStage1DATA.json',
        'glrag-testset/R2AIStage1DATA.json',
    ]
    root = Path(data_dir)
    for name in candidates:
        p = root / name
        if p.exists():
            return p
    jsons = sorted(root.glob('**/*.json'))
    for p in jsons:
        try:
            rows = json.loads(p.read_text(encoding='utf-8'))
            if isinstance(rows, list) and rows and {'id', 'question'}.issubset(rows[0].keys()):
                return p
        except Exception:
            pass
    raise FileNotFoundError('Could not auto-detect questions JSON. Set INPUT_QUERIES manually.')

questions_path = detect_questions_file(DATA_DIR, INPUT_QUERIES)
rows = json.loads(Path(questions_path).read_text(encoding='utf-8'))
print('questions:', questions_path, 'count=', len(rows))

assert SHARD_COUNT >= 1 and 0 <= SHARD_INDEX < SHARD_COUNT
if SHARD_COUNT > 1:
    rows = [r for r in rows if int(r['id']) % SHARD_COUNT == SHARD_INDEX]
    Path(SHARDED_INPUT_PATH).write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    RUN_INPUT = SHARDED_INPUT_PATH
    RESULTS_PATH = f'/content/results_shard{SHARD_INDEX}_of_{SHARD_COUNT}.json'
    SUBMISSION_ZIP_PATH = f'/content/submission_shard{SHARD_INDEX}_of_{SHARD_COUNT}.zip'
    print('shard rows:', len(rows), 'input:', RUN_INPUT)
else:
    RUN_INPUT = str(questions_path)
print('RUN_INPUT =', RUN_INPUT)


## 6. Build fast F2 retrieval pipeline

In [ ]:
from retrieval.kaggle_pipeline import build_unified_pipeline
from retrieval.unified_pipeline import run_dev_set, validate_submission, summarize_timings
from retrieval.doc_anchor import DocAnchorConfig
import retrieval.article_select as article_select
from retrieval.article_select import SelectConfig

# Runtime override in case the cloned branch still has conservative bounds.
article_select.COMPLEXITY_K_BOUNDS.update({
    'simple': (1, 3),
    'medium': (2, 6),
    'complex': (4, 10),
})

anchor_cfg = DocAnchorConfig(
    top_bm25=TOP_BM25,
    top_dense=(TOP_DENSE if USE_DENSE else 0),
    rrf_k=RRF_K,
    top_docs=TOP_DOCS,
    per_doc_articles=PER_DOC_ARTICLES,
    rerank_pool=RERANK_POOL,
    chunks_per_article=CHUNKS_PER_ARTICLE,
)
select_cfg = SelectConfig(
    drop_provincial=DROP_PROVINCIAL,
    max_k=MAX_K,
    min_k=MIN_K,
    rel_margin=REL_MARGIN,
    abs_margin=ABS_MARGIN,
)

t0 = time.time()
pipe, fts = build_unified_pipeline(
    DATA_DIR,
    use_dense=USE_DENSE,
    use_rerank=USE_RERANK,
    use_decomposition=USE_DECOMPOSITION,
    use_llm_decomposition=USE_LLM_DECOMPOSITION,
    use_generator=USE_GENERATOR,
    fts_mode=FTS_MODE,
    gpu_id=GPU_ID,
    anchor_cfg=anchor_cfg,
    select_cfg=select_cfg,
    gen_model=GEN_MODEL,
    gen_load_in_4bit=GEN_LOAD_IN_4BIT,
    max_sub_queries=MAX_SUB_QUERIES,
    gen_context_topk=GEN_CONTEXT_TOPK,
    use_multi_variant=USE_MULTI_VARIANT,
    use_complexity_k=USE_COMPLEXITY_K,
    coverage_quota=COVERAGE_QUOTA,
    max_variants=MAX_VARIANTS,
)
print('pipeline built in %.1fs' % (time.time() - t0))
print({
    'generator': USE_GENERATOR,
    'llm_decomposition': USE_LLM_DECOMPOSITION,
    'max_variants': MAX_VARIANTS,
    'rerank_pool': RERANK_POOL,
    'max_k': MAX_K,
})


## 7. Quick benchmark on 5 queries

In [ ]:
bench_rows = json.loads(Path(RUN_INPUT).read_text(encoding='utf-8'))[:5]
timings = []
for r in bench_rows:
    rec, t = pipe.answer_record_timed(r['id'], r.get('question') or r.get('query') or '')
    timings.append(t)
    print('id=%s k=%d cx=%s nvar=%d total=%.2fs retrieve=%.2fs generate=%.2fs' % (
        r['id'], len(rec['relevant_articles']), t['complexity'], t['n_variants'], t['total'], t['retrieve'], t['generate']))
summary = summarize_timings(timings)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Projected 500 queries: %.1f minutes' % (summary['total_mean'] * 500 / 60.0))


## 8. Full retrieval run → `results.json` → `submission.zip`

In [ ]:
t0 = time.time()
records = run_dev_set(pipe, RUN_INPUT, output_path=RESULTS_PATH)
elapsed = time.time() - t0
print('records:', len(records), 'elapsed_sec:', round(elapsed, 1), 'sec_per_query:', round(elapsed / max(len(records), 1), 2))

expected_ids = [r.get('id') for r in records]
summary = validate_submission(records, expected_ids=expected_ids, require_answer_citation=False)
print('validate:', summary)

with zipfile.ZipFile(SUBMISSION_ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(RESULTS_PATH, arcname='results.json')
print('results:', RESULTS_PATH)
print('submission:', SUBMISSION_ZIP_PATH)

# Free GPU memory.
try:
    import torch, gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass


## 9. Download outputs

In [ ]:
from google.colab import files
files.download(SUBMISSION_ZIP_PATH)
files.download(RESULTS_PATH)
